# Refresh the Bollywood metadata catalogue with TMDB

This notebook refreshes the **metadata seed** through the date you run it. It uses TMDB's paginated movie-discovery API instead of scraping IMDb pages. It merges discoveries with the historical seed and writes a new JSON file. It does **not** modify Home's runtime torrent cache (`latest_bollywood_catalogue.json`) or deploy anything.

TMDB's Hindi-language + India-origin filters are a candidate filter, not a guarantee that every title is Bollywood. Review results before integrating them into the runtime catalogue.

## 1. Configure the API token

In Google Colab, add a secret named `TMDB_READ_ACCESS_TOKEN` (TMDB API Read Access Token). The notebook falls back to a hidden prompt if the secret is not available. Never paste a token into a committed notebook.

In [ ]:
from datetime import date, datetime, timezone
from getpass import getpass
import json, os, re, time
from pathlib import Path
import requests

TODAY = date.today().isoformat()
MAX_PAGES = 500  # TMDB discover pagination cap; lower this for a quick test.
PAGE_DELAY_SECONDS = 0.05
HISTORICAL_SEED_URL = (
    'https://raw.githubusercontent.com/wroxtaaar/home/'
    'feature/background-scraped-catalogue/backend/cache-seed/'
    'bollywood_imdb_historical_metadata.json'
)
OUTPUT_PATH = Path('bollywood_tmdb_updated_metadata.json')

token = ''
try:
    from google.colab import userdata
    token = userdata.get('TMDB_READ_ACCESS_TOKEN') or ''
except Exception:
    pass
if not token:
    token = getpass('TMDB API Read Access Token: ').strip()
if not token:
    raise ValueError('A TMDB Read Access Token is required.')

session = requests.Session()
session.headers.update({'Authorization': f'Bearer {token}', 'accept': 'application/json'})
print(f'Refresh date: {TODAY}; page limit: {MAX_PAGES}')

In [ ]:
def tmdb_get(path, params=None, attempts=6):
    url = f'https://api.themoviedb.org/3/{path.lstrip("/")}'
    for attempt in range(attempts):
        response = session.get(url, params=params or {}, timeout=30)
        if response.status_code == 429 or response.status_code >= 500:
            wait = min(60, int(response.headers.get('Retry-After', 2 ** attempt)))
            print(f'TMDB HTTP {response.status_code}; retrying in {wait}s')
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    raise RuntimeError(f'TMDB request failed after {attempts} attempts: {path}')

response = requests.get(HISTORICAL_SEED_URL, timeout=60)
response.raise_for_status()
historical = response.json()
if not isinstance(historical, list):
    raise ValueError('Historical seed format was unexpected; expected a JSON list.')
print(f'Loaded {len(historical):,} historical rows.')

In [ ]:
# Discover Hindi-language, India-origin movies released no later than today.
# Pagination is explicit and bounded by MAX_PAGES and TMDB's total_pages.
discovered_by_id = {}
page = 1
total_pages = 1
while page <= min(total_pages, MAX_PAGES):
    payload = tmdb_get('discover/movie', {
        'with_original_language': 'hi',
        'with_origin_country': 'IN',
        'primary_release_date.lte': TODAY,
        'sort_by': 'primary_release_date.desc',
        'include_adult': 'false',
        'page': page,
    })
    total_pages = int(payload.get('total_pages') or 1)
    for item in payload.get('results') or []:
        tmdb_id = item.get('id')
        title = (item.get('title') or item.get('original_title') or '').strip()
        release_date = item.get('release_date') or ''
        if not tmdb_id or not title or not re.match(r'^\d{4}-\d{2}-\d{2}$', release_date):
            continue
        if release_date > TODAY or item.get('original_language') != 'hi':
            continue
        record = {
            'tmdbId': int(tmdb_id),
            'imdbId': None,
            'title': title,
            'year': int(release_date[:4]),
            'releaseDate': release_date,
            'genres': [],
            'genreIds': item.get('genre_ids') or [],
            'overview': (item.get('overview') or '').strip(),
            'director': '',
            'cast': [],
            'posterPath': item.get('poster_path'),
            'originalLanguage': item.get('original_language'),
            'originCountry': ['IN'],
            'source': 'TMDB discover/movie',
            'lastRefreshed': TODAY,
        }
        discovered_by_id[record['tmdbId']] = record
    if page == 1 or page % 25 == 0 or page >= min(total_pages, MAX_PAGES):
        print(f'Page {page}/{min(total_pages, MAX_PAGES)}; unique movies: {len(discovered_by_id):,}')
    page += 1
    time.sleep(PAGE_DELAY_SECONDS)

print(f'Collected {len(discovered_by_id):,} unique TMDB movies through {TODAY}.')

In [ ]:
# Merge by TMDB ID when possible, otherwise by normalized title + year.
# Existing historical fields are preserved; new TMDB records remain distinguishable.
def norm_title(value):
    return re.sub(r'[^a-z0-9]+', '', str(value or '').casefold())

merged = []
historical_by_key = {}
for row in historical:
    if not isinstance(row, dict):
        continue
    copy = dict(row)
    merged.append(copy)
    historical_by_key[(norm_title(copy.get('title')), str(copy.get('year') or ''))] = copy

added = 0
enriched = 0
for tmdb_id, record in discovered_by_id.items():
    key = (norm_title(record['title']), str(record['year']))
    existing = historical_by_key.get(key)
    if existing is not None:
        existing.update({k: v for k, v in record.items() if v not in (None, '', [], {})})
        enriched += 1
    else:
        merged.append(record)
        historical_by_key[key] = record
        added += 1

result = {
    'schemaVersion': 1,
    'builtAt': datetime.now(timezone.utc).isoformat(),
    'asOfDate': TODAY,
    'source': 'Historical IMDb-derived seed merged with paginated TMDB discover/movie results',
    'historicalCount': len(historical),
    'tmdbDiscoveredCount': len(discovered_by_id),
    'newRowsAdded': added,
    'historicalRowsEnriched': enriched,
    'results': merged,
}
OUTPUT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Wrote {OUTPUT_PATH.resolve()}')
print(f'Historical rows: {len(historical):,}; TMDB discovered: {len(discovered_by_id):,}; added: {added:,}; enriched: {enriched:,}; final rows: {len(merged):,}')
print('This is a metadata export only; it has not changed Home runtime caches or deployed anything.')

## Before using the output

- The discover filters are not a definitive Bollywood classifier. Review candidates, especially movies with sparse metadata.
- `genreIds` are TMDB genre IDs; `genres`, `director`, and `cast` are not hydrated for new TMDB-only rows in this lightweight refresh.
- Do not copy this output over `latest_bollywood_catalogue.json`: that file has a different schema containing torrent search results.
- If you want this refreshed metadata in Home's runtime catalogue, add a separate validated integration step and tests first. Keep this file as a seed/export until then.